[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Seed-VC_Colab.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on-GitHub-181717?logo=github)](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Seed-VC_Colab.ipynb)

# 🎤 Seed-VC — Zero-Shot Speech & Singing Voice Conversion (DiT Flow-Matching + RMVPE F0 + V2 Style Conversion)

> **Runtime required:** GPU — **T4 (16 GB)**, **L4 (24 GB)**, or **A100 (40/80 GB)**. Both the 22.05 kHz Speech VC and 44.1 kHz Singing Voice Conversion (SVC) pipelines run comfortably on a free Colab T4 GPU (~4–6 GB VRAM).

A complete Google Colab studio for **[Seed-VC](https://github.com/Plachtaa/seed-vc)** (`Plachta/Seed-VC`) — a zero-shot voice conversion and singing voice conversion system combining **Whisper semantic tokenization**, **CAMPPlus speaker style embeddings**, **U-ViT Diffusion Transformer (DiT) conditional flow matching (CFM)**, **RMVPE pitch conditioning**, and **NVIDIA BigVGAN v2 vocoding**.

**Upstream links:** [Plachtaa/seed-vc](https://github.com/Plachtaa/seed-vc) · [Plachta/Seed-VC (HF Models)](https://huggingface.co/Plachta/Seed-VC) · [HF Space](https://huggingface.co/spaces/Plachta/Seed-VC) · [arXiv 2411.09943](https://arxiv.org/abs/2411.09943)

---

## 🎶 Speech VC vs. Singing Voice Conversion (SVC) vs. V2 Style VC

| Mode | Checkpoint (`Plachta/Seed-VC`) | F0 / Pitch Conditioning | Output SR | Best Use Case |
|---|---|---|---|---|
| **1. Zero-Shot Speech VC (`f0_condition=False`)** | `DiT_seed_v2_uvit_whisper_small_wavenet_bigvgan_pruned.pth` | None (`f0_condition=False`) — natural spoken prosody | 22.05 kHz | Dialogue, narration, dubbing, converting any TTS voice into a custom reference speaker |
| **2. Zero-Shot Singing VC (`f0_condition=True`)** | `DiT_seed_v2_uvit_whisper_base_f0_44k_bigvgan_pruned_ft_ema_v2.pth` | **RMVPE F0 (`lj1995/VoiceConversionWebUI`)** + `auto_f0_adjust` + `semi_tone_shift` (`-12..+12`) | **44.1 kHz** | Converting songs/vocals (e.g. from `ACE-Step` or `MiniMax-Music3`) into a custom character voice while preserving melody & vibrato |
| **3. Seed-VC V2 (`configs/v2/vc_wrapper.yaml`)** | AR + CFM V2 Wrapper | Astral Quantization + AR style/emotion transfer (`convert_style=True`) | 22.05 kHz | Transferring accent, speaking style, and emotion alongside speaker timbre |

## 🔗 Cross-Notebook Pipeline (`ACE-Step` / `MiniMax-Music3` $\rightarrow$ `Seed-VC` $\rightarrow$ `InfiniteTalk`)

1. Generate a song in **`ACE-Step_Colab.ipynb`** or **`MiniMax-Music3_Colab.ipynb`** (or speech in **`Chatterbox_Colab.ipynb`**).
2. Select it via the **📂 Cross-Notebook Drive Audio Picker** in this notebook and convert the vocals into your custom character voice.
3. Every converted `.wav` is automatically saved to `/content/drive/MyDrive/AEI_TTS_Out/Seed-VC/`, where **`InfiniteTalk_Colab.ipynb`** and **`LongCat-Video-Avatar-1.5_Colab.ipynb`** immediately discover it to animate a singing or talking portrait!

In [ ]:
# @title STEP 1 — Clone Plachtaa/seed-vc & Install Dependencies
# @markdown Mounts Google Drive, configures `HF_HOME` and `HF_HUB_CACHE` at `/content/drive/MyDrive/AEI_TTS_Cache/Seed-VC`, clones `https://github.com/Plachtaa/seed-vc.git`, and installs runtime dependencies with pinned `gradio>=5.0.0`.

import os, sys, subprocess, shutil, time
import torch

print("=" * 72)
print("🎤 Seed-VC Studio — Environment Setup & Installation")
print("=" * 72)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[GPU] PyTorch {torch.__version__} · Device = {DEVICE}", flush=True)
if DEVICE == "cuda":
    try:
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"[GPU] {gpu_name} ({vram_gb:.1f} GB VRAM)", flush=True)
    except Exception as e:
        print(f"[GPU] Could not query GPU: {e}", flush=True)

MOUNT_GOOGLE_DRIVE = True  # @param {type:"boolean"}
if MOUNT_GOOGLE_DRIVE:
    try:
        if not os.path.exists("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"[Drive] Notice — running with local storage fallback ({e})", flush=True)

if os.path.exists("/content/drive/MyDrive"):
    CACHE_ROOT = "/content/drive/MyDrive/AEI_TTS_Cache/Seed-VC"
    OUT_DIR = "/content/drive/MyDrive/AEI_TTS_Out/Seed-VC"
    VOICES_DIR = "/content/drive/MyDrive/AEI_TTS_Cache/voices"
else:
    CACHE_ROOT = "/content/cache/Seed-VC"
    OUT_DIR = "/content/outputs/Seed-VC"
    VOICES_DIR = "/content/cache/voices"

for d in (CACHE_ROOT, OUT_DIR, VOICES_DIR, "/content/seedvc_out"):
    os.makedirs(d, exist_ok=True)

os.environ.setdefault("HF_HOME", CACHE_ROOT)
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_HUB_CACHE"] = os.path.join(CACHE_ROOT, "hf_cache")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(CACHE_ROOT, "hf_cache")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)

print(f"[Cache] HF_HOME     = {CACHE_ROOT}", flush=True)
print(f"[Out]   OUT_DIR     = {OUT_DIR}", flush=True)
print(f"[Voice] VOICES_DIR  = {VOICES_DIR}", flush=True)

if not shutil.which("ffmpeg"):
    print("[apt] Installing ffmpeg...", flush=True)
    subprocess.run(["apt-get", "-qq", "-y", "install", "ffmpeg"], check=False)

REPO_DIR = "/content/seed-vc"
if not os.path.exists(os.path.join(REPO_DIR, "inference.py")):
    print("\n[git] Cloning https://github.com/Plachtaa/seed-vc.git ...", flush=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Plachtaa/seed-vc.git", REPO_DIR], check=True)
else:
    print(f"\n[git] Repository already present at {REPO_DIR}", flush=True)

# Link /content/seed-vc/checkpoints -> Drive CACHE_ROOT/checkpoints so hf_utils.py persists on Drive
drive_ckpt_dir = os.path.join(CACHE_ROOT, "checkpoints")
os.makedirs(drive_ckpt_dir, exist_ok=True)
repo_ckpt_dir = os.path.join(REPO_DIR, "checkpoints")
if os.path.islink(repo_ckpt_dir):
    pass
elif os.path.isdir(repo_ckpt_dir):
    for fn in os.listdir(repo_ckpt_dir):
        src_p = os.path.join(repo_ckpt_dir, fn)
        dst_p = os.path.join(drive_ckpt_dir, fn)
        if not os.path.exists(dst_p):
            shutil.move(src_p, dst_p)
    shutil.rmtree(repo_ckpt_dir, ignore_errors=True)
    os.symlink(drive_ckpt_dir, repo_ckpt_dir)
else:
    os.symlink(drive_ckpt_dir, repo_ckpt_dir)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

t0 = time.time()
print("\n[pip] Installing Seed-VC runtime dependencies...", flush=True)
PKGS = [
    "munch>=4.0.0",
    "einops>=0.8.0",
    "descript-audio-codec>=1.0.0",
    "pydub>=0.25.1",
    "resemblyzer",
    "jiwer>=3.0.3",
    "hydra-core>=1.3.2",
    "omegaconf",
    "pyyaml",
    "python-dotenv",
    "librosa>=0.10.2",
    "soundfile>=0.12.1",
    "scipy>=1.11.0",
    "transformers>=4.46.0,<5.0.0",
    "accelerate>=0.34.0",
    "huggingface-hub>=0.28.1",
    "tqdm>=4.66.0",
    "gradio>=5.0.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PKGS, check=True)
print(f"[pip] Installed in {time.time() - t0:.1f}s", flush=True)

print("\n[Verify] Checking Seed-VC module imports...", flush=True)
try:
    from modules.commons import build_model, load_checkpoint, recursive_munch
    from hf_utils import load_custom_model_from_hf
    from modules.campplus.DTDNN import CAMPPlus
    from modules.rmvpe import RMVPE
    print("  ✓ Seed-VC modules (build_model, load_checkpoint, CAMPPlus, RMVPE, hf_utils) imported OK!")
except Exception as e:
    print(f"  ✗ Import verification failed: {e}")
    raise

print("\n[Done] STEP 1 complete — proceed to STEP 2.")

In [ ]:
# @title STEP 2 — Pre-Cache Seed-VC Speech (22kHz), Singing (44.1kHz), CAMPPlus & RMVPE Checkpoints
# @markdown Downloads the official checkpoints into `/content/drive/MyDrive/AEI_TTS_Cache/Seed-VC/checkpoints` so both Zero-Shot Speech VC (`f0_condition=False`) and Zero-Shot Singing Voice Conversion (`f0_condition=True`) start immediately.

import os, sys, time
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, snapshot_download

REPO_DIR = "/content/seed-vc"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

CACHE_ROOT = os.environ.get("HF_HOME", "/content/drive/MyDrive/AEI_TTS_Cache/Seed-VC")
os.environ.setdefault("HF_HOME", CACHE_ROOT)
os.environ["HF_HOME"] = CACHE_ROOT

from hf_utils import load_custom_model_from_hf

PRECACHE_SPEECH_VC_22K = True  # @param {type:"boolean"}
PRECACHE_SINGING_SVC_44K = True  # @param {type:"boolean"}
PRECACHE_WHISPER_AND_BIGVGAN = True  # @param {type:"boolean"}

tasks = []
if PRECACHE_SPEECH_VC_22K:
    tasks.append((
        "Speech VC DiT (22.05 kHz)",
        "Plachta/Seed-VC",
        "DiT_seed_v2_uvit_whisper_small_wavenet_bigvgan_pruned.pth",
        "config_dit_mel_seed_uvit_whisper_small_wavenet.yml",
    ))
if PRECACHE_SINGING_SVC_44K:
    tasks.append((
        "Singing SVC DiT (44.1 kHz F0)",
        "Plachta/Seed-VC",
        "DiT_seed_v2_uvit_whisper_base_f0_44k_bigvgan_pruned_ft_ema_v2.pth",
        "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml",
    ))
    tasks.append((
        "RMVPE F0 Pitch Extractor",
        "lj1995/VoiceConversionWebUI",
        "rmvpe.pt",
        None,
    ))

tasks.append((
    "CAMPPlus Speaker Encoder",
    "funasr/campplus",
    "campplus_cn_common.bin",
    None,
))

print(f"[Cache] Downloading {len(tasks)} checkpoint bundle(s) to {os.path.join(CACHE_ROOT, 'checkpoints')} ...")
for label, repo_id, ckpt_fn, cfg_fn in tqdm(tasks, desc="Seed-VC Checkpoints"):
    t0 = time.time()
    try:
        res = load_custom_model_from_hf(repo_id, ckpt_fn, cfg_fn)
        ckpt_p = res[0] if isinstance(res, tuple) else res
        mb = os.path.getsize(ckpt_p) / (1024 ** 2) if os.path.exists(ckpt_p) else 0.0
        print(f"  ✓ [{label}] {ckpt_fn} ({mb:.1f} MB in {time.time() - t0:.1f}s)")
    except Exception as e:
        print(f"  ⚠ [{label}] Warning: {e}")

if PRECACHE_WHISPER_AND_BIGVGAN:
    for hf_repo in [
        "openai/whisper-small",
        "openai/whisper-base",
        "nvidia/bigvgan_v2_22khz_80band_256x",
        "nvidia/bigvgan_v2_44khz_128band_512x",
    ]:
        try:
            t0 = time.time()
            snapshot_download(repo_id=hf_repo)
            print(f"  ✓ Cached {hf_repo} ({time.time() - t0:.1f}s)")
        except Exception as e:
            print(f"  ⚠ Notice caching {hf_repo}: {e}")

print("\n[Done] STEP 2 complete — proceed to STEP 3.")

In [ ]:
# @title STEP 3 — Core Seed-VC Speech & Singing Conversion Engine (Lazy Loader)
# @markdown Implements the exact upstream `Plachtaa/seed-vc` inference pipelines (`inference.py` and `inference_v2.py`):
# @markdown - **Speech VC (`f0_condition=False`, 22.05 kHz)**: Whisper-small encoder + CAMPPlus + U-ViT DiT + BigVGAN 22kHz
# @markdown - **Singing SVC (`f0_condition=True`, 44.1 kHz)**: Whisper-base encoder + RMVPE F0 conditioning + `auto_f0_adjust` + `semi_tone_shift` (`-12..+12` semitones) + BigVGAN 44.1kHz
# @markdown - **Seed-VC V2 (`convert_voice_v2`)**: AR + CFM streaming style/emotion/accent conversion

import os, sys, gc, time, yaml, zipfile, builtins
from pathlib import Path
from typing import Optional, Dict, List, Tuple
import numpy as np
import torch
import torchaudio
import librosa
import soundfile as sf
from tqdm.auto import tqdm

REPO_DIR = "/content/seed-vc"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from modules.commons import build_model, load_checkpoint, recursive_munch
from hf_utils import load_custom_model_from_hf
from modules.campplus.DTDNN import CAMPPlus
from modules.audio import mel_spectrogram

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_ROOT = os.environ.get("HF_HOME", "/content/drive/MyDrive/AEI_TTS_Cache/Seed-VC")
os.environ.setdefault("HF_HOME", CACHE_ROOT)

OUT_DIR = "/content/drive/MyDrive/AEI_TTS_Out/Seed-VC" if os.path.exists("/content/drive/MyDrive") else "/content/outputs/Seed-VC"
VOICES_DIR = "/content/drive/MyDrive/AEI_TTS_Cache/voices" if os.path.exists("/content/drive/MyDrive") else "/content/cache/voices"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(VOICES_DIR, exist_ok=True)
os.makedirs("/content/seedvc_out", exist_ok=True)

# Cache of loaded pipelines keyed by ('speech_22k' or 'singing_44k')
SEEDVC_PIPELINES: Dict[str, dict] = {}
SEEDVC_V2_WRAPPER = None


def free_seedvc_vram() -> str:
    global SEEDVC_V2_WRAPPER
    SEEDVC_PIPELINES.clear()
    SEEDVC_V2_WRAPPER = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, tot_b = torch.cuda.mem_get_info(0)
        return f"Loaded pipelines: none · GPU free: {free_b/1e9:.1f}/{tot_b/1e9:.1f} GB"
    return "Loaded pipelines: none (CPU)"


def get_seedvc_pipeline(f0_condition: bool = False) -> dict:
    """Lazily load either the 22.05kHz Speech VC pipeline (f0_condition=False)
    or the 44.1kHz Singing Voice Conversion pipeline (f0_condition=True)."""
    key = "singing_44k" if f0_condition else "speech_22k"
    if key in SEEDVC_PIPELINES and SEEDVC_PIPELINES[key] is not None:
        return SEEDVC_PIPELINES[key]

    # Free the other pipeline first on smaller GPUs
    if SEEDVC_PIPELINES:
        SEEDVC_PIPELINES.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    t0 = time.time()
    print(f"[Seed-VC] Loading {'Singing SVC (44.1 kHz, F0=True)' if f0_condition else 'Speech VC (22.05 kHz, F0=False)'} on {DEVICE} ...", flush=True)

    if not f0_condition:
        dit_ckpt_path, dit_cfg_path = load_custom_model_from_hf(
            "Plachta/Seed-VC",
            "DiT_seed_v2_uvit_whisper_small_wavenet_bigvgan_pruned.pth",
            "config_dit_mel_seed_uvit_whisper_small_wavenet.yml",
        )
        f0_fn = None
    else:
        dit_ckpt_path, dit_cfg_path = load_custom_model_from_hf(
            "Plachta/Seed-VC",
            "DiT_seed_v2_uvit_whisper_base_f0_44k_bigvgan_pruned_ft_ema_v2.pth",
            "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml",
        )
        from modules.rmvpe import RMVPE
        rmvpe_path = load_custom_model_from_hf("lj1995/VoiceConversionWebUI", "rmvpe.pt", None)
        f0_extractor = RMVPE(rmvpe_path, is_half=False, device=DEVICE)
        f0_fn = f0_extractor.infer_from_audio

    config = yaml.safe_load(open(dit_cfg_path, "r"))
    model_params = recursive_munch(config["model_params"])
    model_params.dit_type = "DiT"
    model = build_model(model_params, stage="DiT")
    hop_length = config["preprocess_params"]["spect_params"]["hop_length"]
    sr = config["preprocess_params"]["sr"]

    model, _, _, _ = load_checkpoint(
        model,
        None,
        dit_ckpt_path,
        load_only_params=True,
        ignore_modules=[],
        is_distributed=False,
    )
    for k in model:
        model[k].eval()
        model[k].to(DEVICE)
    model.cfm.estimator.setup_caches(max_batch_size=1, max_seq_length=8192)

    campplus_ckpt_path = load_custom_model_from_hf("funasr/campplus", "campplus_cn_common.bin", config_filename=None)
    campplus_model = CAMPPlus(feat_dim=80, embedding_size=192)
    campplus_model.load_state_dict(torch.load(campplus_ckpt_path, map_location="cpu"))
    campplus_model.eval().to(DEVICE)

    vocoder_type = model_params.vocoder.type
    if vocoder_type == "bigvgan":
        from modules.bigvgan import bigvgan
        bigvgan_name = model_params.vocoder.name
        bigvgan_model = bigvgan.BigVGAN.from_pretrained(bigvgan_name, use_cuda_kernel=False)
        bigvgan_model.remove_weight_norm()
        vocoder_fn = bigvgan_model.eval().to(DEVICE)
    elif vocoder_type == "hifigan":
        from modules.hifigan.generator import HiFTGenerator
        from modules.hifigan.f0_predictor import ConvRNNF0Predictor
        hift_config = yaml.safe_load(open("configs/hifigan.yml", "r"))
        hift_gen = HiFTGenerator(**hift_config["hift"], f0_predictor=ConvRNNF0Predictor(**hift_config["f0_predictor"]))
        hift_path = load_custom_model_from_hf("FunAudioLLM/CosyVoice-300M", "hift.pt", None)
        hift_gen.load_state_dict(torch.load(hift_path, map_location="cpu"))
        vocoder_fn = hift_gen.eval().to(DEVICE)
    else:
        raise ValueError(f"Unsupported vocoder_type: {vocoder_type}")

    from transformers import AutoFeatureExtractor, WhisperModel
    whisper_name = model_params.speech_tokenizer.name
    whisper_model = WhisperModel.from_pretrained(whisper_name, torch_dtype=torch.float16).to(DEVICE)
    del whisper_model.decoder
    whisper_feature_extractor = AutoFeatureExtractor.from_pretrained(whisper_name)

    def semantic_fn(waves_16k):
        ori_inputs = whisper_feature_extractor(
            [waves_16k.squeeze(0).cpu().numpy()],
            return_tensors="pt",
            return_attention_mask=True,
        )
        ori_input_features = whisper_model._mask_input_features(
            ori_inputs.input_features, attention_mask=ori_inputs.attention_mask
        ).to(DEVICE)
        with torch.no_grad():
            ori_outputs = whisper_model.encoder(
                ori_input_features.to(whisper_model.encoder.dtype),
                head_mask=None,
                output_attentions=False,
                output_hidden_states=False,
                return_dict=True,
            )
        S_ori = ori_outputs.last_hidden_state.to(torch.float32)
        S_ori = S_ori[:, :waves_16k.size(-1) // 320 + 1]
        return S_ori

    mel_fn_args = {
        "n_fft": config["preprocess_params"]["spect_params"]["n_fft"],
        "win_size": config["preprocess_params"]["spect_params"]["win_length"],
        "hop_size": config["preprocess_params"]["spect_params"]["hop_length"],
        "num_mels": config["preprocess_params"]["spect_params"]["n_mels"],
        "sampling_rate": sr,
        "fmin": config["preprocess_params"]["spect_params"].get("fmin", 0),
        "fmax": None if config["preprocess_params"]["spect_params"].get("fmax", "None") == "None" else 8000,
        "center": False,
    }
    to_mel = lambda x: mel_spectrogram(x, **mel_fn_args)

    bundle = {
        "model": model,
        "semantic_fn": semantic_fn,
        "f0_fn": f0_fn,
        "vocoder_fn": vocoder_fn,
        "campplus_model": campplus_model,
        "mel_fn": to_mel,
        "mel_fn_args": mel_fn_args,
        "sr": sr,
        "hop_length": hop_length,
        "f0_condition": f0_condition,
    }
    SEEDVC_PIPELINES[key] = bundle
    print(f"[Seed-VC] '{key}' ready in {time.time() - t0:.1f}s (sr={sr} Hz, hop={hop_length})", flush=True)
    return bundle


def adjust_f0_semitones(f0_sequence: torch.Tensor, n_semitones: float) -> torch.Tensor:
    factor = 2.0 ** (float(n_semitones) / 12.0)
    return f0_sequence * factor


def crossfade(chunk1: np.ndarray, chunk2: np.ndarray, overlap: int) -> np.ndarray:
    fade_out = np.cos(np.linspace(0, np.pi / 2, overlap)) ** 2
    fade_in = np.cos(np.linspace(np.pi / 2, 0, overlap)) ** 2
    if len(chunk2) < overlap:
        chunk2[:overlap] = chunk2[:overlap] * fade_in[:len(chunk2)] + (chunk1[-overlap:] * fade_out)[:len(chunk2)]
    else:
        chunk2[:overlap] = chunk2[:overlap] * fade_in + chunk1[-overlap:] * fade_out
    return chunk2


@torch.no_grad()
def run_seed_vc(
    source_path: str,
    target_path: str,
    f0_condition: bool = False,
    auto_f0_adjust: bool = True,
    semi_tone_shift: int = 0,
    diffusion_steps: int = 25,
    length_adjust: float = 1.0,
    inference_cfg_rate: float = 0.7,
    fp16: bool = True,
    prefix: str = "seedvc",
) -> Dict[str, object]:
    """Run zero-shot Speech VC (f0_condition=False, 22.05kHz) or Singing Voice Conversion (f0_condition=True, 44.1kHz)."""
    if not source_path or not os.path.exists(str(source_path)):
        raise FileNotFoundError(f"Source audio not found: {source_path}")
    if not target_path or not os.path.exists(str(target_path)):
        raise FileNotFoundError(f"Target reference voice not found: {target_path}")

    pipe = get_seedvc_pipeline(f0_condition=bool(f0_condition))
    model = pipe["model"]
    semantic_fn = pipe["semantic_fn"]
    f0_fn = pipe["f0_fn"]
    vocoder_fn = pipe["vocoder_fn"]
    campplus_model = pipe["campplus_model"]
    mel_fn = pipe["mel_fn"]

    sr = 22050 if not f0_condition else 44100
    hop_length = 256 if not f0_condition else 512
    max_context_window = sr // hop_length * 30
    overlap_frame_len = 16
    overlap_wave_len = overlap_frame_len * hop_length

    t0 = time.time()
    source_audio = librosa.load(str(source_path), sr=sr)[0]
    ref_audio = librosa.load(str(target_path), sr=sr)[0]

    source_audio_t = torch.tensor(source_audio).unsqueeze(0).float().to(DEVICE)
    ref_audio_t = torch.tensor(ref_audio[:sr * 25]).unsqueeze(0).float().to(DEVICE)

    converted_waves_16k = torchaudio.functional.resample(source_audio_t, sr, 16000)
    if converted_waves_16k.size(-1) <= 16000 * 30:
        S_alt = semantic_fn(converted_waves_16k)
    else:
        overlapping_time = 5
        S_alt_list = []
        buffer = None
        traversed_time = 0
        while traversed_time < converted_waves_16k.size(-1):
            if buffer is None:
                chunk = converted_waves_16k[:, traversed_time:traversed_time + 16000 * 30]
            else:
                chunk = torch.cat(
                    [buffer, converted_waves_16k[:, traversed_time:traversed_time + 16000 * (30 - overlapping_time)]],
                    dim=-1,
                )
            S_chunk = semantic_fn(chunk)
            if traversed_time == 0:
                S_alt_list.append(S_chunk)
            else:
                S_alt_list.append(S_chunk[:, 50 * overlapping_time:])
            buffer = chunk[:, -16000 * overlapping_time:]
            traversed_time += 30 * 16000 if traversed_time == 0 else chunk.size(-1) - 16000 * overlapping_time
        S_alt = torch.cat(S_alt_list, dim=1)

    ori_waves_16k = torchaudio.functional.resample(ref_audio_t, sr, 16000)
    S_ori = semantic_fn(ori_waves_16k)

    mel = mel_fn(source_audio_t.float())
    mel2 = mel_fn(ref_audio_t.float())

    target_lengths = torch.LongTensor([int(mel.size(2) * float(length_adjust))]).to(mel.device)
    target2_lengths = torch.LongTensor([mel2.size(2)]).to(mel2.device)

    feat2 = torchaudio.compliance.kaldi.fbank(
        ori_waves_16k, num_mel_bins=80, dither=0, sample_frequency=16000
    )
    feat2 = feat2 - feat2.mean(dim=0, keepdim=True)
    style2 = campplus_model(feat2.unsqueeze(0))

    if f0_condition:
        F0_ori = f0_fn(ori_waves_16k[0], thred=0.03)
        F0_alt = f0_fn(converted_waves_16k[0], thred=0.03)
        F0_ori = torch.from_numpy(F0_ori).to(DEVICE)[None]
        F0_alt = torch.from_numpy(F0_alt).to(DEVICE)[None]

        voiced_F0_ori = F0_ori[F0_ori > 1]
        voiced_F0_alt = F0_alt[F0_alt > 1]

        log_f0_alt = torch.log(F0_alt + 1e-5)
        shifted_log_f0_alt = log_f0_alt.clone()
        if auto_f0_adjust and voiced_F0_ori.numel() > 0 and voiced_F0_alt.numel() > 0:
            median_log_f0_ori = torch.median(torch.log(voiced_F0_ori + 1e-5))
            median_log_f0_alt = torch.median(torch.log(voiced_F0_alt + 1e-5))
            shifted_log_f0_alt[F0_alt > 1] = log_f0_alt[F0_alt > 1] - median_log_f0_alt + median_log_f0_ori
        shifted_f0_alt = torch.exp(shifted_log_f0_alt)
        if int(semi_tone_shift) != 0:
            shifted_f0_alt[F0_alt > 1] = adjust_f0_semitones(shifted_f0_alt[F0_alt > 1], int(semi_tone_shift))
    else:
        F0_ori = None
        shifted_f0_alt = None

    cond, _, _, _, _ = model.length_regulator(
        S_alt, ylens=target_lengths, n_quantizers=3, f0=shifted_f0_alt
    )
    prompt_condition, _, _, _, _ = model.length_regulator(
        S_ori, ylens=target2_lengths, n_quantizers=3, f0=F0_ori
    )

    max_source_window = max(64, max_context_window - mel2.size(2))
    processed_frames = 0
    generated_wave_chunks = []

    while processed_frames < cond.size(1):
        chunk_cond = cond[:, processed_frames:processed_frames + max_source_window]
        is_last_chunk = processed_frames + max_source_window >= cond.size(1)
        cat_condition = torch.cat([prompt_condition, chunk_cond], dim=1)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16 if (fp16 and DEVICE.type == "cuda") else torch.float32):
            vc_target = model.cfm.inference(
                cat_condition,
                torch.LongTensor([cat_condition.size(1)]).to(mel2.device),
                mel2,
                style2,
                None,
                int(diffusion_steps),
                inference_cfg_rate=float(inference_cfg_rate),
            )
            vc_target = vc_target[:, :, mel2.size(-1):]
        vc_wave = vocoder_fn(vc_target.float()).squeeze()
        if vc_wave.ndim == 1:
            vc_wave = vc_wave[None, :]
        if processed_frames == 0:
            if is_last_chunk:
                generated_wave_chunks.append(vc_wave[0].cpu().numpy())
                break
            generated_wave_chunks.append(vc_wave[0, :-overlap_wave_len].cpu().numpy())
            previous_chunk = vc_wave[0, -overlap_wave_len:]
            processed_frames += vc_target.size(2) - overlap_frame_len
        elif is_last_chunk:
            generated_wave_chunks.append(crossfade(previous_chunk.cpu().numpy(), vc_wave[0].cpu().numpy(), overlap_wave_len))
            processed_frames += vc_target.size(2) - overlap_frame_len
            break
        else:
            generated_wave_chunks.append(crossfade(previous_chunk.cpu().numpy(), vc_wave[0, :-overlap_wave_len].cpu().numpy(), overlap_wave_len))
            previous_chunk = vc_wave[0, -overlap_wave_len:]
            processed_frames += vc_target.size(2) - overlap_frame_len

    full_audio = np.concatenate(generated_wave_chunks).astype(np.float32)
    full_audio = np.clip(full_audio, -1.0, 1.0)
    elapsed = time.time() - t0
    dur = len(full_audio) / sr

    src_stem = Path(source_path).stem[:24]
    tgt_stem = Path(target_path).stem[:24]
    mode_tag = f"svc44k_st{int(semi_tone_shift):+d}" if f0_condition else "vc22k"
    fname = f"{prefix}_{mode_tag}_{src_stem}_to_{tgt_stem}_{int(time.time())}.wav"
    out_path = os.path.join(OUT_DIR, fname)
    sf.write(out_path, full_audio, sr, subtype="PCM_16")
    return {
        "wav_path": out_path,
        "audio_np": full_audio,
        "sr": sr,
        "duration_s": dur,
        "elapsed_s": elapsed,
        "rtf": elapsed / max(dur, 1e-3),
        "f0_condition": f0_condition,
    }


def discover_aei_drive_audio(max_files: int = 80) -> Tuple[List[str], List[str]]:
    """Return (source_audio_candidates, reference_voice_candidates) across AEI Drive folders and built-in examples."""
    src_roots = [
        OUT_DIR,
        "/content/drive/MyDrive/AEI_TTS_Out",
        "/content/drive/MyDrive/AEI_3D_Out/ACE-Step",
        "/content/drive/MyDrive/AEI_3D_Out/MiniMax-Music3",
        "/content/drive/MyDrive/AEI_3D_Out",
        os.path.join(REPO_DIR, "examples", "source"),
    ]
    ref_roots = [
        VOICES_DIR,
        os.path.join(REPO_DIR, "examples", "reference"),
        "/content/drive/MyDrive/AEI_TTS_Out",
    ]
    exts = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

    def _scan(roots):
        items, seen = [], set()
        for r in roots:
            if not os.path.isdir(r):
                continue
            for dp, _, fs in os.walk(r):
                for fn in fs:
                    if os.path.splitext(fn)[1].lower() in exts:
                        p = os.path.abspath(os.path.join(dp, fn))
                        if p not in seen:
                            seen.add(p)
                            try:
                                items.append((os.path.getmtime(p), p))
                            except OSError:
                                pass
        items.sort(key=lambda x: x[0], reverse=True)
        return [p for _, p in items[:max_files]]

    return _scan(src_roots), _scan(ref_roots)


builtins.run_seed_vc = run_seed_vc
builtins.discover_aei_drive_audio = discover_aei_drive_audio
builtins.free_seedvc_vram = free_seedvc_vram
builtins.SEEDVC_OUT_DIR = OUT_DIR

src_list, ref_list = discover_aei_drive_audio()
print(f"[Core] STEP 3 ready. Found {len(src_list)} source clip(s) and {len(ref_list)} reference voice(s).")
print(f"       Outputs saved to: {OUT_DIR}")

In [ ]:
# @title STEP 4 — Launch Gradio UI (Speech VC, Singing SVC + Pitch Shift & Cross-Notebook Drive Picker)
# @markdown Launches a 4-Tab Gradio Studio:
# @markdown 1. **Tab 1: Zero-Shot Speech Voice Conversion (`f0_condition=False`, 22.05 kHz)**
# @markdown 2. **Tab 2: Zero-Shot Singing Voice Conversion (`f0_condition=True`, 44.1 kHz RMVPE + Pitch Shift `-12..+12` Semitones)**
# @markdown 3. **Tab 3: 📂 Cross-Notebook Drive Audio & Voice Explorer**
# @markdown 4. **Tab 4: Batch Directory / Playlist Voice Conversion**

import os, time, zipfile
import gradio as gr
from tqdm.auto import tqdm
from IPython.display import clear_output

src_init, ref_init = discover_aei_drive_audio(max_files=80)
SRC_CHOICES = ["(Use uploaded file above)"] + src_init
REF_CHOICES = ["(Use uploaded file above)"] + ref_init


def _resolve_path(up_path, drive_pick):
    if up_path and os.path.exists(str(up_path)):
        return str(up_path)
    if drive_pick and drive_pick != "(Use uploaded file above)" and os.path.exists(str(drive_pick)):
        return str(drive_pick)
    return None


def _refresh_pickers():
    s_list, r_list = discover_aei_drive_audio(max_files=80)
    s_ch = ["(Use uploaded file above)"] + s_list
    r_ch = ["(Use uploaded file above)"] + r_list
    return (
        gr.update(choices=s_ch, value=s_ch[1] if len(s_ch) > 1 else s_ch[0]),
        gr.update(choices=r_ch, value=r_ch[1] if len(r_ch) > 1 else r_ch[0]),
        gr.update(choices=s_ch, value=s_ch[1] if len(s_ch) > 1 else s_ch[0]),
        gr.update(choices=r_ch, value=r_ch[1] if len(r_ch) > 1 else r_ch[0]),
    )


def _ui_speech_vc(src_up, src_dr, ref_up, ref_dr, steps, len_adj, cfg_rate, fp16, progress=gr.Progress()):
    src_p = _resolve_path(src_up, src_dr)
    ref_p = _resolve_path(ref_up, ref_dr)
    if not src_p:
        raise gr.Error("Upload a Source Speech clip or pick one from the Cross-Notebook Drive Audio Picker.")
    if not ref_p:
        raise gr.Error("Upload a Target Reference Voice (5–25s) or pick one from the Drive Voice Picker.")
    try:
        progress(0.15, desc="Running Zero-Shot Speech VC (22.05 kHz, f0_condition=False)...")
        res = run_seed_vc(
            source_path=src_p,
            target_path=ref_p,
            f0_condition=False,
            auto_f0_adjust=False,
            semi_tone_shift=0,
            diffusion_steps=int(steps),
            length_adjust=float(len_adj),
            inference_cfg_rate=float(cfg_rate),
            fp16=bool(fp16),
            prefix="seedvc_speech",
        )
        progress(1.0, desc="Done!")
        meta = (
            f"Mode: Zero-Shot Speech VC (f0_condition=False · 22.05 kHz BigVGAN)\n"
            f"Source: {os.path.basename(src_p)} -> Target: {os.path.basename(ref_p)}\n"
            f"Duration: {res['duration_s']:.2f}s · Wall: {res['elapsed_s']:.1f}s (RTF={res['rtf']:.2f}) · Steps: {int(steps)}\n"
            f"Saved to Drive: {res['wav_path']}"
        )
        return (res["sr"], res["audio_np"]), res["wav_path"], meta
    except Exception as e:
        raise gr.Error(f"Speech VC failed: {e}")


def _ui_singing_svc(src_up, src_dr, ref_up, ref_dr, auto_f0, semitones, steps, len_adj, cfg_rate, fp16, progress=gr.Progress()):
    src_p = _resolve_path(src_up, src_dr)
    ref_p = _resolve_path(ref_up, ref_dr)
    if not src_p:
        raise gr.Error("Upload a Source Song/Vocal track or pick one from the Cross-Notebook Drive Audio Picker.")
    if not ref_p:
        raise gr.Error("Upload a Target Reference Voice or pick one from the Drive Voice Picker.")
    try:
        progress(0.15, desc="Running Singing Voice Conversion (44.1 kHz RMVPE, f0_condition=True)...")
        res = run_seed_vc(
            source_path=src_p,
            target_path=ref_p,
            f0_condition=True,
            auto_f0_adjust=bool(auto_f0),
            semi_tone_shift=int(semitones),
            diffusion_steps=int(steps),
            length_adjust=float(len_adj),
            inference_cfg_rate=float(cfg_rate),
            fp16=bool(fp16),
            prefix="seedvc_singing",
        )
        progress(1.0, desc="Done!")
        meta = (
            f"Mode: Zero-Shot Singing Voice Conversion (f0_condition=True · 44.1 kHz RMVPE + BigVGAN)\n"
            f"Source Song: {os.path.basename(src_p)} -> Target Singer: {os.path.basename(ref_p)}\n"
            f"Auto F0 Adjust: {bool(auto_f0)} · Pitch Shift: {int(semitones):+d} semitones · Steps: {int(steps)}\n"
            f"Duration: {res['duration_s']:.2f}s · Wall: {res['elapsed_s']:.1f}s (RTF={res['rtf']:.2f})\n"
            f"Saved to Drive (ready for InfiniteTalk!): {res['wav_path']}"
        )
        return (res["sr"], res["audio_np"]), res["wav_path"], meta
    except Exception as e:
        raise gr.Error(f"Singing VC failed: {e}")


def _ui_batch_convert(input_dir, ref_up, ref_dr, is_singing, auto_f0, semitones, steps, cfg_rate, progress=gr.Progress()):
    ref_p = _resolve_path(ref_up, ref_dr)
    if not ref_p:
        raise gr.Error("Provide a Target Reference Voice (upload or Drive picker).")
    if not input_dir or not os.path.isdir(str(input_dir)):
        raise gr.Error(f"Input directory not found: {input_dir}")
    exts = (".wav", ".mp3", ".flac", ".m4a", ".ogg")
    files = sorted(f for f in os.listdir(input_dir) if f.lower().endswith(exts))
    if not files:
        raise gr.Error(f"No audio files found in {input_dir}")

    zip_path = os.path.join(OUT_DIR, f"seedvc_batch_{int(time.time())}.zip")
    logs = []
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for i, fname in enumerate(tqdm(files, desc="Seed-VC Batch")):
            progress((i + 1) / len(files), desc=f"Converting {i+1}/{len(files)}: {fname}")
            try:
                r = run_seed_vc(
                    source_path=os.path.join(input_dir, fname),
                    target_path=ref_p,
                    f0_condition=bool(is_singing),
                    auto_f0_adjust=bool(auto_f0),
                    semi_tone_shift=int(semitones),
                    diffusion_steps=int(steps),
                    inference_cfg_rate=float(cfg_rate),
                    prefix=f"batch_{i+1:03d}",
                )
                zf.write(r["wav_path"], arcname=os.path.basename(r["wav_path"]))
                logs.append(f"[{i+1:03d}/{len(files)}] ✓ {fname} -> {r['duration_s']:.2f}s ({r['elapsed_s']:.1f}s)")
            except Exception as e:
                logs.append(f"[{i+1:03d}/{len(files)}] ✗ {fname}: {e}")
    return zip_path, "\n".join(logs)


with gr.Blocks(title="Seed-VC — Zero-Shot Speech & Singing Voice Conversion", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎤 Seed-VC Studio — Zero-Shot Speech & Singing Voice Conversion\n"
        "Convert spoken dialogue (`22.05 kHz`) or full songs (`44.1 kHz` with RMVPE melody preservation & semitone transposition) into any reference speaker's voice."
    )
    welcome_md = gr.Markdown()

    with gr.Row():
        refresh_all_btn = gr.Button("🔄 Refresh Cross-Notebook Drive Audio & Voice Pickers", size="sm")

    with gr.Tabs():
        # ── TAB 1: Zero-Shot Speech VC ──
        with gr.Tab("1. Zero-Shot Speech Voice Conversion (f0_condition=False)"):
            with gr.Row():
                with gr.Column(scale=3):
                    s1_src_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Source Speech Audio (upload or record)")
                    s1_src_dr = gr.Dropdown(
                        choices=SRC_CHOICES,
                        value=SRC_CHOICES[1] if len(SRC_CHOICES) > 1 else SRC_CHOICES[0],
                        label="📂 Cross-Notebook Drive Audio Picker (Source Speech from AEI_TTS_Out / examples)",
                        info="Select speech generated by Chatterbox, Kokoro, Qwen3-TTS, OpenVoice, or built-in examples.",
                    )
                    s1_ref_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Target Reference Voice (5–25s clean speech)")
                    s1_ref_dr = gr.Dropdown(
                        choices=REF_CHOICES,
                        value=REF_CHOICES[1] if len(REF_CHOICES) > 1 else REF_CHOICES[0],
                        label="📂 Cross-Notebook Drive Voice Picker (Target Reference Voice from AEI_TTS_Cache/voices)",
                        info="Select any reference voice stored on Google Drive or Seed-VC built-in reference voices.",
                    )
                with gr.Column(scale=2):
                    s1_steps = gr.Slider(10, 50, value=25, step=1, label="Diffusion Steps", info="10–15 = fast preview, 25 = official high-quality default.")
                    s1_len = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Length Adjust (Speaking Rate)", info="<1.0 = faster speech, >1.0 = slower speech, 1.0 = original pacing.")
                    s1_cfg = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Inference CFG Rate", info="Controls speaker timbre adherence (default 0.7).")
                    s1_fp16 = gr.Checkbox(value=True, label="Use FP16 Autocast", info="2x faster inference on T4/L4/A100 GPUs.")
                    s1_btn = gr.Button("🗣️ Convert Speech Voice", variant="primary")
                    s1_out_aud = gr.Audio(label="Converted Speech Output (22.05 kHz)", type="numpy")
                    s1_out_file = gr.File(label="Download Converted WAV")
                    s1_meta = gr.Textbox(label="Conversion Summary", lines=4, interactive=False)

            s1_btn.click(
                _ui_speech_vc,
                inputs=[s1_src_up, s1_src_dr, s1_ref_up, s1_ref_dr, s1_steps, s1_len, s1_cfg, s1_fp16],
                outputs=[s1_out_aud, s1_out_file, s1_meta],
            )

        # ── TAB 2: Zero-Shot Singing Voice Conversion (SVC) ──
        with gr.Tab("2. Zero-Shot Singing Voice Conversion (f0_condition=True, 44.1 kHz)"):
            gr.Markdown(
                "Convert any vocal track or song (including songs generated in **`ACE-Step_Colab.ipynb`** or **`MiniMax-Music3_Colab.ipynb`**) "
                "into any target speaker's voice while preserving melody, vibrato, and harmonics via **RMVPE F0 conditioning** at **44.1 kHz**!"
            )
            with gr.Row():
                with gr.Column(scale=3):
                    s2_src_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Source Singing / Song Audio (upload vocal track or song)")
                    s2_src_dr = gr.Dropdown(
                        choices=SRC_CHOICES,
                        value=SRC_CHOICES[1] if len(SRC_CHOICES) > 1 else SRC_CHOICES[0],
                        label="📂 Cross-Notebook Drive Song/Vocal Picker (ACE-Step / MiniMax-Music3 / AEI_TTS_Out)",
                        info="Automatically finds songs from ACE-Step, MiniMax-Music3, and Seed-VC examples.",
                    )
                    s2_ref_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Target Singer / Reference Voice (spoken or sung 5–25s clip)")
                    s2_ref_dr = gr.Dropdown(
                        choices=REF_CHOICES,
                        value=REF_CHOICES[1] if len(REF_CHOICES) > 1 else REF_CHOICES[0],
                        label="📂 Cross-Notebook Drive Target Voice Picker",
                        info="Even a spoken reference clip can be used to sing the source melody!",
                    )
                with gr.Column(scale=2):
                    s2_autof0 = gr.Checkbox(
                        value=True,
                        label="Auto F0 Pitch Alignment (auto_f0_adjust)",
                        info="Aligns median log-F0 of source song to target voice's natural register (disable for singing -> singing when key must stay fixed).",
                    )
                    s2_semi = gr.Slider(
                        -12, 12, value=0, step=1,
                        label="Semitone Pitch Shift (semi_tone_shift)",
                        info="Transpose singing pitch in semitones (-12 = 1 octave down, 0 = original key, +12 = 1 octave up).",
                    )
                    s2_steps = gr.Slider(10, 50, value=25, step=1, label="Diffusion Steps", info="25–30 steps recommended for 44.1 kHz singing vocals.")
                    s2_len = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Length Adjust", info="Keep at 1.0 to preserve exact musical tempo and beat sync.")
                    s2_cfg = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Inference CFG Rate", info="Default 0.7 balances vocal clarity and target timbre similarity.")
                    s2_fp16 = gr.Checkbox(value=True, label="Use FP16 Autocast", info="Accelerates 44.1 kHz DiT inference.")
                    s2_btn = gr.Button("🎶 Convert Singing Voice (44.1 kHz SVC)", variant="primary")
                    s2_out_aud = gr.Audio(label="Converted Singing Output (44.1 kHz)", type="numpy")
                    s2_out_file = gr.File(label="Download Converted Song WAV")
                    s2_meta = gr.Textbox(label="SVC Summary", lines=5, interactive=False)

            s2_btn.click(
                _ui_singing_svc,
                inputs=[s2_src_up, s2_src_dr, s2_ref_up, s2_ref_dr, s2_autof0, s2_semi, s2_steps, s2_len, s2_cfg, s2_fp16],
                outputs=[s2_out_aud, s2_out_file, s2_meta],
            )

        # ── TAB 3: Cross-Notebook Drive Audio Explorer ──
        with gr.Tab("3. 📂 Cross-Notebook Drive Audio Previewer"):
            with gr.Row():
                with gr.Column():
                    ex_pick = gr.Dropdown(
                        choices=src_init + [p for p in ref_init if p not in src_init],
                        value=src_init[0] if src_init else None,
                        label="Select Any Discovered Drive / Example Audio File to Preview",
                        info="Browses AEI_TTS_Out, AEI_3D_Out (ACE-Step, MiniMax-Music3), AEI_TTS_Cache/voices, and Seed-VC examples.",
                    )
                    ex_btn = gr.Button("▶️ Load & Preview Selected File")
                with gr.Column():
                    ex_audio = gr.Audio(label="Audio Preview", type="filepath")
            ex_btn.click(fn=lambda p: p if (p and os.path.exists(str(p))) else None, inputs=[ex_pick], outputs=[ex_audio])

        # ── TAB 4: Batch Folder Conversion ──
        with gr.Tab("4. Batch Folder Conversion"):
            with gr.Row():
                with gr.Column():
                    b_dir = gr.Textbox(
                        value="/content/seed-vc/examples/source",
                        label="Source Audio Directory",
                        info="Folder containing .wav/.mp3/.flac files to convert.",
                    )
                    b_ref_up = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Target Reference Voice")
                    b_ref_dr = gr.Dropdown(
                        choices=REF_CHOICES,
                        value=REF_CHOICES[1] if len(REF_CHOICES) > 1 else REF_CHOICES[0],
                        label="📂 Or Pick Target Voice from Drive",
                        info="Reference speaker applied to every file in the batch.",
                    )
                    b_sing = gr.Checkbox(value=False, label="Singing Mode (f0_condition=True, 44.1 kHz)", info="Enable for singing vocal stems; leave unchecked for spoken audio.")
                    b_autof0 = gr.Checkbox(value=True, label="Auto F0 Adjust", info="Aligns median pitch in Singing mode.")
                    b_semi = gr.Slider(-12, 12, value=0, step=1, label="Semitone Shift", info="Pitch transposition in Singing mode.")
                    b_steps = gr.Slider(10, 50, value=25, step=1, label="Diffusion Steps", info="Default 25.")
                    b_cfg = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Inference CFG Rate", info="Default 0.7.")
                    b_btn = gr.Button("📦 Run Batch Voice Conversion", variant="primary")
                with gr.Column():
                    b_zip = gr.File(label="Download Converted Batch (.zip)")
                    b_log = gr.Textbox(label="Batch Log", lines=10, interactive=False)

            b_btn.click(
                _ui_batch_convert,
                inputs=[b_dir, b_ref_up, b_ref_dr, b_sing, b_autof0, b_semi, b_steps, b_cfg],
                outputs=[b_zip, b_log],
            )

    refresh_all_btn.click(_refresh_pickers, outputs=[s1_src_dr, s1_ref_dr, s2_src_dr, s2_ref_dr])

    def _welcome():
        return (
            f"✅ **Seed-VC Studio Ready** on `{DEVICE}` · Converted outputs are automatically saved to `{OUT_DIR}` "
            f"for instant use in `InfiniteTalk_Colab.ipynb` and `LongCat-Video-Avatar-1.5_Colab.ipynb`."
        )

    demo.load(_welcome, inputs=None, outputs=[welcome_md])

clear_output()
demo.queue(concurrency_limit=1).launch(share=True, inline=False, show_error=True)

In [ ]:
# @title STEP 5 — Keep Colab Session Alive
# @markdown Prevents Google Colab from disconnecting during interactive Gradio sessions or long batch conversions.

import threading, time

def _seedvc_keepalive():
    while True:
        time.sleep(60)

threading.Thread(target=_seedvc_keepalive, daemon=True).start()
print("[Keep-Alive] Heartbeat thread running. Output directory:", OUT_DIR)

In [ ]:
# @title STEP 6 — Quick Test (Speech VC or Singing SVC without UI)
# @markdown Run a single zero-shot Speech VC (`F0_CONDITION = False`) or Singing Voice Conversion (`F0_CONDITION = True`) call using built-in examples or custom paths.

F0_CONDITION = True  # @param {type:"boolean"}
AUTO_F0_ADJUST = True  # @param {type:"boolean"}
SEMI_TONE_SHIFT = 0  # @param {type:"slider", min:-12, max:12, step:1}
DIFFUSION_STEPS = 25  # @param {type:"slider", min:10, max:50, step:1}
LENGTH_ADJUST = 1.0  # @param {type:"slider", min:0.5, max:2.0, step:0.05}
INFERENCE_CFG_RATE = 0.7  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
SOURCE_AUDIO_PATH = ""  # @param {type:"string"}
TARGET_VOICE_PATH = ""  # @param {type:"string"}

import os
from IPython.display import Audio, FileLink, display

src_candidates, ref_candidates = discover_aei_drive_audio()
src_path = SOURCE_AUDIO_PATH.strip() or (src_candidates[0] if src_candidates else "/content/seed-vc/examples/source/source_s1.wav")
tgt_path = TARGET_VOICE_PATH.strip() or (ref_candidates[0] if ref_candidates else "/content/seed-vc/examples/reference/s1p1.wav")

print(f"[QuickTest] Mode={'Singing SVC (44.1kHz)' if F0_CONDITION else 'Speech VC (22.05kHz)'}")
print(f"  Source : {src_path}")
print(f"  Target : {tgt_path}")

res = run_seed_vc(
    source_path=src_path,
    target_path=tgt_path,
    f0_condition=F0_CONDITION,
    auto_f0_adjust=AUTO_F0_ADJUST,
    semi_tone_shift=SEMI_TONE_SHIFT,
    diffusion_steps=DIFFUSION_STEPS,
    length_adjust=LENGTH_ADJUST,
    inference_cfg_rate=INFERENCE_CFG_RATE,
    fp16=True,
    prefix="quicktest",
)

print(f"✓ Saved: {res['wav_path']} ({res['duration_s']:.2f}s @ {res['sr']} Hz in {res['elapsed_s']:.1f}s, RTF={res['rtf']:.2f})")
display(Audio(res["wav_path"], autoplay=False))
display(FileLink(res["wav_path"]))

In [ ]:
# @title STEP 7 — Batch Voice Conversion (Directory Queue with tqdm)
# @markdown Converts all audio files in `SOURCE_DIR` to the timbre of `TARGET_VOICE_PATH` with per-file `try`/`except` error handling and `tqdm` progress reporting.

SOURCE_DIR = "/content/seed-vc/examples/source"  # @param {type:"string"}
TARGET_VOICE_PATH = ""  # @param {type:"string"}
F0_CONDITION = False  # @param {type:"boolean"}
AUTO_F0_ADJUST = True  # @param {type:"boolean"}
SEMI_TONE_SHIFT = 0  # @param {type:"slider", min:-12, max:12, step:1}
DIFFUSION_STEPS = 25  # @param {type:"slider", min:10, max:50, step:1}
INFERENCE_CFG_RATE = 0.7  # @param {type:"slider", min:0.0, max:1.0, step:0.05}

import os, time
from tqdm.auto import tqdm
from IPython.display import Audio, FileLink, display

_, ref_candidates = discover_aei_drive_audio()
tgt_path = TARGET_VOICE_PATH.strip() or (ref_candidates[0] if ref_candidates else "/content/seed-vc/examples/reference/s1p1.wav")

if not os.path.isdir(SOURCE_DIR):
    raise SystemExit(f"SOURCE_DIR not found: {SOURCE_DIR}")

exts = (".wav", ".mp3", ".flac", ".m4a", ".ogg")
batch_files = sorted(f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(exts))
print(f"[Batch] Converting {len(batch_files)} file(s) from {SOURCE_DIR} -> target={os.path.basename(tgt_path)}")

outputs = []
for idx, fname in enumerate(tqdm(batch_files, desc="Step 7 Seed-VC Batch")):
    in_p = os.path.join(SOURCE_DIR, fname)
    try:
        r = run_seed_vc(
            source_path=in_p,
            target_path=tgt_path,
            f0_condition=F0_CONDITION,
            auto_f0_adjust=AUTO_F0_ADJUST,
            semi_tone_shift=SEMI_TONE_SHIFT,
            diffusion_steps=DIFFUSION_STEPS,
            inference_cfg_rate=INFERENCE_CFG_RATE,
            prefix=f"batch_{idx+1:03d}",
        )
        outputs.append(r["wav_path"])
        print(f"  [{idx+1:03d}/{len(batch_files)}] ✓ {fname} -> {r['wav_path']} ({r['duration_s']:.2f}s)")
    except Exception as e:
        print(f"  [{idx+1:03d}/{len(batch_files)}] ✗ FAILED {fname}: {e}")

print(f"\n[Done] {len(outputs)}/{len(batch_files)} converted in {OUT_DIR}")
if outputs:
    display(Audio(outputs[-1], autoplay=False))
    display(FileLink(outputs[-1]))